# RNNs and LSTMs from Scratch - HW3

**Total Points: 100**

In this assignment, you will implement RNNs and LSTMs from scratch using PyTorch. You will complete the missing code sections marked with `#TODO` comments. Each section is worth the indicated points.

## Learning Objectives
1. Understand the architecture of RNNs and LSTMs
2. Implement forward passes for recurrent architectures
3. Train and evaluate sequence models
4. Apply RNNs/LSTMs to text classification and language modeling tasks

## Grading Breakdown
- Part 1: RNN Cell Implementation (20 points)
- Part 2: RNN with Embedding Forward Pass (15 points)
- Part 3: Stacked RNN Implementation (20 points)
- Part 4: LSTM Cell Implementation (25 points)
- Part 5: Language Model Implementation (20 points)

**Instructions:** Replace all `#TODO` sections with your implementation. Make sure to read the hints carefully!

RNNs are recurrent neural networks, a class of neural networks that is well-suited for processing sequential data. They are widely used in various applications such as natural language processing, time series prediction, and speech recognition.

LSTMs, or Long Short-Term Memory networks, are a type of RNN that addresses the vanishing gradient problem encountered in traditional RNNs. They are capable of learning long-term dependencies in sequential data by using special units called memory cells.

In this assignment, we will implement RNNs and LSTMs from scratch using PyTorch (autograd takes care of the backward pass). We will cover the following topics:

1. Understanding the architecture of RNNs and LSTMs
2. Implementing the forward pass
3. Training the models
4. Evaluating the models

Let's get started!

## 1. Understanding the Architecture

### RNN Architecture

An RNN processes input sequences one element at a time while maintaining a hidden state that captures information about previous elements in the sequence. The basic equations for an RNN are:

- Hidden state update:
  $$
  h_t = \tanh(W_{hh} h_{t-1} + W_{xh} x_t + b_h)
  $$
  
- Output:
  $$
  y_t = W_{hy} h_t + b_y
  $$

Where:
- $h_t$ is the hidden state at time step $t$
- $x_t$ is the input at time step $t$
- $W_{hh}$, $W_{xh}$, and $W_{hy}$ are weight matrices
- $b_h$ and $b_y$ are bias vectors

### LSTM Architecture

An LSTM consists of several components: input gate, forget gate, output gate, and cell state. The equations for an LSTM are:

- Forget gate:
  $$
  f_t = \sigma(W_f [h_{t-1}, x_t] + b_f)
  $$

- Input gate:
  $$
  i_t = \sigma(W_i [h_{t-1}, x_t] + b_i)
  $$
  
- Candidate cell state:
  $$
  \tilde{C}_t = \tanh(W_C [h_{t-1}, x_t] + b_C)
  $$

- Cell state update:
  $$
  C_t = f_t \odot C_{t-1} + i_t \odot \tilde{C}_t
  $$

- Output gate:
  $$
  o_t = \sigma(W_o [h_{t-1}, x_t] + b_o)
  $$

- Hidden state:
  $$
  h_t = o_t \odot \tanh(C_t)
  $$

Where:
- $[h_{t-1}, x_t]$ is the concatenation of the previous hidden state and the current input
- $\sigma$ is the sigmoid function and $\odot$ is element-wise multiplication
- $C_t$ is the cell state (long-term memory)

In Part 4 you will implement the same equations in the equivalent form with separate input-to-hidden and hidden-to-hidden weights ($W_{ih} x_t + W_{hh} h_{t-1}$ instead of $W [h_{t-1}, x_t]$).

## Setup and Data Loading
The following cells handle imports and data loading. **You don't need to modify these cells.**

In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision import models
from typing import List, Tuple
import math
import numpy as np

import datasets



In [2]:
data = datasets.load_dataset("fancyzhx/ag_news", split="train[:10%]")
data_val = datasets.load_dataset("fancyzhx/ag_news", split="test")

In [3]:
import re
from collections import Counter

def tokenize_basic(text: str):
    # lowercase + keep alnum + simple contractions
    return re.findall(r"[a-z0-9]+(?:'[a-z0-9]+)?", text.lower())

# Build vocab from TRAIN TEXT ONLY
train_texts = list(data["text"])

def build_vocab(texts, max_vocab_size=30000, min_freq=2):
    counter = Counter()
    for t in texts:
        counter.update(tokenize_basic(t))
    # keep words meeting min_freq, capped by max_vocab_size
    most_common = [w for (w, c) in counter.most_common() if c >= min_freq][:max_vocab_size]
    vocab = ["<pad>", "<unk>"] + most_common
    word2idx = {w: i for i, w in enumerate(vocab)}
    return vocab, word2idx

vocab, word2idx = build_vocab(train_texts, max_vocab_size=30000, min_freq=2)

class Tokenizer:
    def __init__(self, vocab, word2idx):
        self.vocab = vocab
        self.word2idx = word2idx
        self.idx2word = {idx: word for word, idx in word2idx.items()}
        self.pad_token = "<pad>"
        self.unk_token = "<unk>"
        self.pad_idx = self.word2idx[self.pad_token]

    def encode(self, text: str):
        toks = tokenize_basic(text)
        return [self.word2idx.get(w, self.word2idx[self.unk_token]) for w in toks]

    def decode(self, indices):
        return " ".join([self.idx2word.get(i, self.unk_token) for i in indices])

tokenizer = Tokenizer(vocab, word2idx)


class Dataset(torch.utils.data.Dataset):
    def __init__(self, data, tokenizer: Tokenizer, max_length: int = 50):
        self.data = data
        self.tokenizer = tokenizer
        self.max_length = max_length
        self.pad_idx = tokenizer.pad_idx

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        text = self.data[idx]["text"]
        label = self.data[idx]["label"]
        ids = self.tokenizer.encode(text)
        length = min(len(ids), self.max_length)
        ids = ids[:self.max_length]
        if len(ids) < self.max_length:
            ids = ids + [self.pad_idx] * (self.max_length - len(ids))
        return torch.tensor(ids, dtype=torch.long), torch.tensor(label, dtype=torch.long), torch.tensor(length, dtype=torch.long)


In [4]:
class LanguageModelDataset(torch.utils.data.Dataset):
    """Dataset for language modeling tasks (no labels needed)"""
    def __init__(self, data, tokenizer: Tokenizer, max_length: int = 50):
        self.data = data
        self.tokenizer = tokenizer
        self.max_length = max_length
        self.pad_idx = tokenizer.pad_idx

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        text = self.data[idx]["text"]
        # Skip empty texts
        if not text.strip():
            text = "empty text"
        ids = self.tokenizer.encode(text)
        length = min(len(ids), self.max_length)
        ids = ids[:self.max_length]
        if len(ids) < self.max_length:
            ids = ids + [self.pad_idx] * (self.max_length - len(ids))
        # For language modeling, we return dummy label and length
        return torch.tensor(ids, dtype=torch.long), torch.tensor(0, dtype=torch.long), torch.tensor(length, dtype=torch.long)

## Part 1: RNN Cell Implementation (20 points)

Implement the RNN cell from scratch. The RNN cell computes:

$$h_t = \tanh(W_{ih} \cdot x_t + b_{ih} + W_{hh} \cdot h_{t-1} + b_{hh})$$

Where:
- $x_t$ is the input at time step $t$
- $h_{t-1}$ is the previous hidden state
- $W_{ih}$, $W_{hh}$ are weight matrices
- $b_{ih}$, $b_{hh}$ are bias vectors

In [5]:
class RNN_cell(nn.Module):
    def __init__(self, input_size: int, hidden_size: int):
        super().__init__()
        self.input_size = input_size
        self.hidden_size = hidden_size

        # TODO: Define the weights and biases (part of Part 1)
        # Hint 1: Wrap each tensor in nn.Parameter so it is registered with the module,
        #         initialized by init_weights() below, and updated by the optimizer
        # Hint 2: F.linear(x, W, b) computes x @ W.T + b, so W has shape (out_features, in_features)
        # Hint 3: W_ih: (hidden_size, input_size), W_hh: (hidden_size, hidden_size), b_ih and b_hh: (hidden_size,)
        # Hint 4: Create an uninitialized tensor with torch.Tensor(rows, cols), e.g. nn.Parameter(torch.Tensor(hidden_size, input_size))
        #         (LSTMCell in Part 4 uses the same pattern with 4 * hidden_size)
        #TODO: self.W_ih = ...
        #TODO: self.W_hh = ...
        #TODO: self.b_ih = ...
        #TODO: self.b_hh = ...
        raise NotImplementedError("Define the RNN cell parameters")

        self.init_weights()

    def init_weights(self):
        stdv = 1.0 / math.sqrt(self.hidden_size)
        for weight in self.parameters():
            weight.data.uniform_(-stdv, stdv)

    def forward(self, x: torch.Tensor, h_prev: torch.Tensor) -> torch.Tensor:
        # TODO: Implement the RNN cell forward pass (20 points)
        # Hint 1: Use F.linear(input, weight, bias) to compute linear transformations
        # Hint 2: Apply F.linear to x with W_ih and b_ih
        # Hint 3: Apply F.linear to h_prev with W_hh and b_hh
        # Hint 4: Add the two results together
        # Hint 5: Apply tanh activation: torch.tanh()
        # Your code should be a single return statement combining all these operations

        #TODO: return torch.tanh(...)
        raise NotImplementedError("Implement the RNN cell forward pass")


## Part 2: RNN with Embedding Forward Pass (15 points)

Complete the forward pass of the RNN model with embedding layer. The model should:
1. Embed the input tokens
2. Process each time step through the RNN cell
3. Extract the hidden state at the last valid token position
4. Pass it through a fully connected layer for classification

In [6]:
class RNN_with_embedding(nn.Module):
    def __init__(self, max_token_len: int, vocab_size: int, embedding_dim: int, hidden_size: int, output_size: int, pad_idx: int):
        super().__init__()
        # Embedding layer (padding_idx freezes PAD embedding to zeros)
        self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=pad_idx)
        self.rnn_cell = RNN_cell(embedding_dim, hidden_size)
        self.fc = nn.Linear(hidden_size, output_size)
        self.max_token_len = max_token_len
        self.hidden_size = hidden_size

    def forward(self, x: torch.Tensor, lengths: torch.Tensor) -> torch.Tensor:
        emb = self.embedding(x)  # (B, L, E)
        B, L, E = emb.size()

        # TODO: Initialize hidden state (3 points)
        # Hint: Create a zero tensor of shape (B, self.hidden_size) on the same device as emb
        # Use: torch.zeros(..., device=emb.device)
        #TODO: h = ...
        raise NotImplementedError("Initialize hidden state")

        hs = []

        # TODO: Unroll RNN through time (7 points)
        # Hint 1: Loop through each time step: for t in range(L)
        # Hint 2: Get input at time t: emb[:, t] which has shape (B, E)
        # Hint 3: Pass through RNN cell: h = self.rnn_cell(input_t, h)
        # Hint 4: Store hidden state: hs.append(h.unsqueeze(1))
        #TODO: Implement time loop
        raise NotImplementedError("Implement RNN unrolling")

        Hseq = torch.cat(hs, dim=1)  # (B, L, H)

        # TODO: Extract last valid hidden state (5 points)
        # Hint 1: Get indices of last valid tokens: idx = (lengths - 1).clamp_min(0)
        # Hint 2: Create batch indices: batch_indices = torch.arange(B, device=emb.device)
        # Hint 3: Index into Hseq: last_h = Hseq[batch_indices, idx]
        #TODO: idx = ...
        #TODO: batch_indices = ...
        #TODO: last_h = ...
        raise NotImplementedError("Extract last hidden state")

        return self.fc(last_h)


In [7]:
# Model hyperparameters
max_token_len = 128
vocab_size = len(vocab)
embedding_dim = 50
hidden_size = 100
output_size = 4  # 4 classes in AG News
pad_idx = tokenizer.pad_idx

# Instantiate model
model = RNN_with_embedding(max_token_len, vocab_size, embedding_dim, hidden_size, output_size, pad_idx)
print(model)

# Create data loaders
dataset = Dataset(data, tokenizer, max_length=max_token_len)
dataset_val = Dataset(data_val, tokenizer, max_length=max_token_len)
dataloader = torch.utils.data.DataLoader(dataset, batch_size=32, shuffle=True)
dataloader_val = torch.utils.data.DataLoader(dataset_val, batch_size=32, shuffle=False)


NotImplementedError: Define the RNN cell parameters

## Training Functions
The following training and validation functions are provided. **You don't need to modify these.**

In [ ]:
from tqdm import tqdm

def validate(model, dataloader_val, criterion, device):
    model.eval()
    total_loss, correct, total = 0.0, 0, 0
    with torch.no_grad():
        for inputs, labels, lengths in tqdm(dataloader_val):
            inputs, labels, lengths = inputs.to(device), labels.to(device), lengths.to(device)
            outputs = model(inputs, lengths)
            loss = criterion(outputs, labels)
            total_loss += loss.item()
            preds = outputs.argmax(dim=1)
            total += labels.size(0)
            correct += (preds == labels).sum().item()
    return total_loss / len(dataloader_val), 100.0 * correct / total

def train(model, dataloader, dataloader_val, num_epochs: int = 5, learning_rate: float = 0.001):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model.to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)
    history = {"train_loss": [], "val_loss": [], "val_accuracy": []}

    for epoch in range(num_epochs):
        model.train()
        total_loss = 0.0
        for inputs, labels, lengths in tqdm(dataloader):
            inputs, labels, lengths = inputs.to(device), labels.to(device), lengths.to(device)
            optimizer.zero_grad()
            outputs = model(inputs, lengths)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            total_loss += loss.item()

        avg_train_loss = total_loss / len(dataloader)
        val_loss, val_acc = validate(model, dataloader_val, criterion, device)
        history["train_loss"].append(avg_train_loss)
        history["val_loss"].append(val_loss)
        history["val_accuracy"].append(val_acc)
        print(f"Epoch [{epoch+1}/{num_epochs}] Train: {avg_train_loss:.4f} Val: {val_loss:.4f} Acc: {val_acc:.2f}%")
    return history

# Train the model (uncomment after implementing the above TODOs)
# history = train(model, dataloader, dataloader_val, num_epochs=5, learning_rate=0.001)


## Part 3: Stacked/Multi-layer RNN (20 points)

Implement a stacked RNN architecture where the output of one RNN layer becomes the input to the next layer. This allows the model to learn more complex hierarchical representations.

**Note:** the 3-layer RNN can train unstably: its validation accuracy may jump around between epochs or drop in the last epochs. This does not by itself mean your implementation is wrong. If you want to dig into why, see the **Bonus** at the end of the notebook.

In [ ]:
class RNNWithStackedLayers(nn.Module):
    def __init__(self, max_token_len: int, vocab_size: int, embedding_dim: int,
                 hidden_size: int, output_size: int, pad_idx: int, num_layers: int = 2):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=pad_idx)

        # TODO: Create a list of RNN cells for each layer (8 points)
        # Hint 1: Use nn.ModuleList to store multiple RNN cells
        # Hint 2: First layer takes embedding_dim as input_size
        # Hint 3: Subsequent layers take hidden_size as input_size
        # Hint 4: All layers have hidden_size as output size
        # Example structure: [RNN_cell(embedding_dim, hidden_size), RNN_cell(hidden_size, hidden_size), ...]
        #TODO: self.rnn_cells = nn.ModuleList([...])
        raise NotImplementedError("Create RNN cells for each layer")

        self.fc = nn.Linear(hidden_size, output_size)
        self.max_token_len = max_token_len
        self.hidden_size = hidden_size
        self.num_layers = num_layers

    def forward(self, x: torch.Tensor, lengths: torch.Tensor) -> torch.Tensor:
        emb = self.embedding(x)  # (B, L, E)
        B, L, E = emb.size()

        # TODO: Initialize hidden states for all layers (4 points)
        # Hint: Create a list of num_layers hidden states, each of shape (B, hidden_size)
        # Use list comprehension: [torch.zeros(...) for _ in range(self.num_layers)]
        #TODO: h = [...]
        raise NotImplementedError("Initialize hidden states for all layers")

        # TODO: Initialize list to store hidden states for each layer (1 point)
        # Hint: Create a list of empty lists: [[] for _ in range(self.num_layers)]
        #TODO: hs = [...]
        raise NotImplementedError("Initialize hs list")

        # TODO: Unroll through time with stacked layers (7 points)
        # Hint 1: Outer loop: for t in range(L)
        # Hint 2: Set input_t = emb[:, t] initially
        # Hint 3: Inner loop: for layer in range(self.num_layers)
        # Hint 4: Update hidden state: h[layer] = self.rnn_cells[layer](input_t, h[layer])
        # Hint 5: Store hidden state: hs[layer].append(h[layer].unsqueeze(1))
        # Hint 6: Update input_t for next layer: input_t = h[layer]
        #TODO: Implement nested loops
        raise NotImplementedError("Implement stacked RNN unrolling")

        # Use last layer's hidden states
        Hseq = hs[-1]
        Hseq = torch.cat(Hseq, dim=1)  # (B, L, H)

        # Extract last valid hidden state
        idx = (lengths - 1).clamp_min(0)
        batch_indices = torch.arange(B, device=emb.device)
        last_h = Hseq[batch_indices, idx]

        return self.fc(last_h)

# Instantiate stacked RNN
model_stacked = RNNWithStackedLayers(max_token_len, vocab_size, embedding_dim,
                                      hidden_size, output_size, pad_idx, num_layers=3)
print(model_stacked)

# Train the model (uncomment after implementing the above TODOs)
# history_stacked = train(model_stacked, dataloader, dataloader_val, num_epochs=5, learning_rate=0.001)


## Part 4: LSTM Cell Implementation (25 points)

Implement an LSTM cell from scratch. LSTMs use gates to control information flow:

- **Input gate** ($i_t$): Controls what new information to add
- **Forget gate** ($f_t$): Controls what information to discard
- **Output gate** ($o_t$): Controls what to output
- **Cell state** ($c_t$): Long-term memory

The LSTM equations are:
$$
\begin{align}
gates &= W_{ih} \cdot x_t + b_{ih} + W_{hh} \cdot h_{t-1} + b_{hh} \\
i_t, f_t, g_t, o_t &= \text{split}(gates, 4) \\
i_t &= \sigma(i_t), \quad f_t = \sigma(f_t), \quad o_t = \sigma(o_t) \\
g_t &= \tanh(g_t) \\
c_t &= f_t \odot c_{t-1} + i_t \odot g_t \\
h_t &= o_t \odot \tanh(c_t)
\end{align}
$$

In [ ]:
class LSTMCell(nn.Module):
    def __init__(self, input_size: int, hidden_size: int):
        super().__init__()
        self.input_size = input_size
        self.hidden_size = hidden_size

        # Weights for all 4 gates combined (input, forget, cell, output)
        self.W_ih = nn.Parameter(torch.Tensor(4 * hidden_size, input_size))
        self.W_hh = nn.Parameter(torch.Tensor(4 * hidden_size, hidden_size))
        self.b_ih = nn.Parameter(torch.Tensor(4 * hidden_size))
        self.b_hh = nn.Parameter(torch.Tensor(4 * hidden_size))
        self.init_weights()

    def init_weights(self):
        stdv = 1.0 / math.sqrt(self.hidden_size)
        for weight in self.parameters():
            weight.data.uniform_(-stdv, stdv)

    def forward(self, x: torch.Tensor, h_prev: torch.Tensor, c_prev: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor]:
        # TODO: Compute combined gates (5 points)
        # Hint: gates = F.linear(x, self.W_ih, self.b_ih) + F.linear(h_prev, self.W_hh, self.b_hh)
        #TODO: gates = ...
        raise NotImplementedError("Compute gates")

        # TODO: Split gates into input, forget, cell, output gates (4 points)
        # Hint: Use gates.chunk(4, 1) to split into 4 equal parts along dimension 1
        # This returns a tuple: (i, f, g, o)
        #TODO: i, f, g, o = ...
        raise NotImplementedError("Split gates")

        # TODO: Apply activations to gates (8 points - 2 each)
        # Hint 1: Apply sigmoid to i: i = torch.sigmoid(i)
        # Hint 2: Apply sigmoid to f: f = torch.sigmoid(f)
        # Hint 3: Apply tanh to g: g = torch.tanh(g)
        # Hint 4: Apply sigmoid to o: o = torch.sigmoid(o)
        #TODO: i = ...
        #TODO: f = ...
        #TODO: g = ...
        #TODO: o = ...
        raise NotImplementedError("Apply activations")

        # TODO: Update cell state (4 points)
        # Hint: c = f * c_prev + i * g
        # The forget gate controls what to keep from c_prev
        # The input gate controls what to add from g
        #TODO: c = ...
        raise NotImplementedError("Update cell state")

        # TODO: Compute hidden state (4 points)
        # Hint: h = o * torch.tanh(c)
        # The output gate controls what to output from the cell state
        #TODO: h = ...
        raise NotImplementedError("Compute hidden state")

        return h, c


In [ ]:
class LSTMWithEmbedding(nn.Module):
    def __init__(self, max_token_len: int, vocab_size: int, embedding_dim: int,
                 hidden_size: int, output_size: int, pad_idx: int):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=pad_idx)
        self.lstm_cell = LSTMCell(embedding_dim, hidden_size)
        self.fc = nn.Linear(hidden_size, output_size)
        self.max_token_len = max_token_len
        self.hidden_size = hidden_size

    def forward(self, x: torch.Tensor, lengths: torch.Tensor) -> torch.Tensor:
        emb = self.embedding(x)  # (B, L, E)
        B, L, E = emb.size()

        # Initialize hidden and cell states
        h = torch.zeros(B, self.hidden_size, device=emb.device)
        c = torch.zeros(B, self.hidden_size, device=emb.device)
        hs = []

        # Unroll LSTM through time
        for t in range(L):
            h, c = self.lstm_cell(emb[:, t], h, c)
            hs.append(h.unsqueeze(1))

        Hseq = torch.cat(hs, dim=1)  # (B, L, H)

        # Extract last valid hidden state
        idx = (lengths - 1).clamp_min(0)
        batch_indices = torch.arange(B, device=emb.device)
        last_h = Hseq[batch_indices, idx]

        return self.fc(last_h)

# Instantiate LSTM model
model_lstm = LSTMWithEmbedding(max_token_len, vocab_size, embedding_dim, hidden_size, output_size, pad_idx)
print(model_lstm)

# Train the model (uncomment after implementing the LSTM cell)
# history_lstm = train(model_lstm, dataloader, dataloader_val, num_epochs=5, learning_rate=0.001)


## Part 5: LSTM Language Model (20 points)

Implement an LSTM-based language model that predicts the next token in a sequence. Unlike classification, this model outputs a probability distribution over the vocabulary for each token position.

In [ ]:
class LSTMLanguageModel(nn.Module):
    def __init__(self, vocab_size: int, embedding_dim: int, hidden_size: int, pad_idx: int):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=pad_idx)
        self.lstm_cell = LSTMCell(embedding_dim, hidden_size)

        # TODO: Create output layer for language modeling (3 points)
        # Hint: The output layer should map from hidden_size to vocab_size
        # This produces logits for each token in the vocabulary
        #TODO: self.fc = nn.Linear(..., ...)
        raise NotImplementedError("Create output layer")

        self.hidden_size = hidden_size
        self.vocab_size = vocab_size

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # TODO: Get embeddings (2 points)
        # Hint: emb = self.embedding(x)
        #TODO: emb = ...
        raise NotImplementedError("Get embeddings")

        B, L, E = emb.size()

        # TODO: Initialize hidden and cell states (3 points)
        # Hint: Create two zero tensors of shape (B, self.hidden_size)
        #TODO: h = ...
        #TODO: c = ...
        raise NotImplementedError("Initialize states")

        outputs = []

        # TODO: Unroll LSTM and collect outputs at each time step (7 points)
        # Hint 1: Loop through time: for t in range(L)
        # Hint 2: Update LSTM: h, c = self.lstm_cell(emb[:, t], h, c)
        # Hint 3: Get output logits: out = self.fc(h)
        # Hint 4: Store output: outputs.append(out.unsqueeze(1))
        #TODO: Implement time loop
        raise NotImplementedError("Unroll LSTM")

        # TODO: Concatenate outputs (2 points)
        # Hint: output_seq = torch.cat(outputs, dim=1)
        # This creates a tensor of shape (B, L, vocab_size)
        #TODO: output_seq = ...
        raise NotImplementedError("Concatenate outputs")

        return output_seq


In [ ]:
def train_language_model(model, dataloader, dataloader_val, num_epochs: int = 10, learning_rate: float = 0.001):
    """Training function for language modeling."""
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model.to(device)

    # TODO: Define loss function for language modeling (3 points)
    # Hint 1: Use nn.CrossEntropyLoss()
    # Hint 2: Set ignore_index=tokenizer.pad_idx to ignore padding tokens
    # This is important so we don't penalize the model for predicting padding
    #TODO: criterion = nn.CrossEntropyLoss(ignore_index=...)
    raise NotImplementedError("Define loss function")

    optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)
    history = {"train_loss": [], "val_loss": [], "perplexity": []}

    for epoch in range(num_epochs):
        model.train()
        total_loss = 0.0

        for inputs, _, _ in tqdm(dataloader, desc=f"Epoch {epoch+1}/{num_epochs} [Train]"):
            inputs = inputs.to(device)

            # Create targets by shifting input by 1 position
            targets = torch.cat([inputs[:, 1:], torch.zeros(inputs.size(0), 1, dtype=torch.long, device=device)], dim=1)

            optimizer.zero_grad()
            outputs = model(inputs)  # (B, L, V)

            # Reshape for loss computation
            outputs_flat = outputs.view(-1, model.vocab_size)
            targets_flat = targets.view(-1)

            loss = criterion(outputs_flat, targets_flat)
            loss.backward()
            optimizer.step()
            total_loss += loss.item()

        # Validation
        model.eval()
        val_loss = 0.0
        with torch.no_grad():
            for inputs, _, _ in tqdm(dataloader_val, desc=f"Epoch {epoch+1}/{num_epochs} [Val]"):
                inputs = inputs.to(device)
                targets = torch.cat([inputs[:, 1:], torch.zeros(inputs.size(0), 1, dtype=torch.long, device=device)], dim=1)

                outputs = model(inputs)
                outputs_flat = outputs.view(-1, model.vocab_size)
                targets_flat = targets.view(-1)

                loss = criterion(outputs_flat, targets_flat)
                val_loss += loss.item()

        avg_train_loss = total_loss / len(dataloader)
        avg_val_loss = val_loss / len(dataloader_val)
        perplexity = torch.exp(torch.tensor(avg_val_loss)).item()

        history["train_loss"].append(avg_train_loss)
        history["val_loss"].append(avg_val_loss)
        history["perplexity"].append(perplexity)

        print(f"Epoch [{epoch+1}/{num_epochs}] | Train Loss: {avg_train_loss:.4f} | Val Loss: {avg_val_loss:.4f} | Val Perplexity: {perplexity:.2f}")

    return history

# Create language modeling dataset
lm_dataset = LanguageModelDataset(data, tokenizer, max_length=max_token_len)
lm_dataset_val = LanguageModelDataset(data_val, tokenizer, max_length=max_token_len)
lm_dataloader = torch.utils.data.DataLoader(lm_dataset, batch_size=32, shuffle=True)
lm_dataloader_val = torch.utils.data.DataLoader(lm_dataset_val, batch_size=32, shuffle=False)

# Instantiate and train (uncomment after implementing the above TODOs)
# language_model = LSTMLanguageModel(vocab_size, embedding_dim, hidden_size, pad_idx)
# history_lm = train_language_model(language_model, lm_dataloader, lm_dataloader_val, num_epochs=5, learning_rate=0.001)


## Bonus: Stabilize the Stacked RNN

If you trained the 3-layer stacked RNN in Part 3, you may have noticed that its training is unstable: the validation accuracy jumps around between epochs and can drop sharply in the last epochs, sometimes ending up worse than the single-layer RNN from Part 2. This happens even with a correct implementation.

Your task is to figure out why, and fix it.

1. **Diagnose** find out what goes wrong during training. Support your answer with evidence from your own experiments (something you measured or printed while training), not just a guess.
2. **Fix** change *how the model is trained* so that the 3-layer stacked RNN trains reliably. Rules:
   - Keep the Part 3 model as it is: `RNNWithStackedLayers` with `num_layers=3` and the same `embedding_dim`, `hidden_size`, data and 5 epochs.
   - Do not edit the provided `train()` function above (Parts 2-4 rely on it). Write your own training function in the cell below instead.
   - Show the per-epoch validation accuracy of the original and your fixed training side by side. A single run can be lucky, so compare more than one random seed (`torch.manual_seed(...)` before creating each model).
3. **Explain (1 point):** in a few sentences, explain why your fix works.


In [ ]:
# Bonus: your code here (diagnosis experiments, your training function, and the comparison)


**Bonus answer (diagnosis and explanation):**

*Your answer here*


## Submission Instructions

### Before Submitting
1. Make sure you have implemented all TODO sections
2. Uncomment the training code and run all cells
3. Verify that your models train successfully
4. Check that you don't have any NotImplementedError exceptions

### What to Submit
- This completed Jupyter notebook (.ipynb file)
- A PDF export of the notebook with all outputs visible

### Grading Rubric
| Part | Description | Points |
|------|-------------|--------|
| 1 | RNN Cell forward pass implementation | 20 |
| 2 | RNN with Embedding forward pass | 15 |
| 3 | Stacked RNN implementation | 20 |
| 4 | LSTM Cell implementation | 25 |
| 5 | Language Model implementation | 20 |
| **Total** | | **100** |

### Tips for Success
- Read all hints carefully - they guide you through the implementation
- Pay attention to tensor shapes mentioned in comments
- Test each part incrementally before moving to the next
- Compare your validation accuracy with expected ranges:
  - RNN: ~60-70% accuracy on AG News
  - LSTM: ~65-75% accuracy on AG News
- Use `print()` statements to debug tensor shapes if needed
- Runtime: the classifiers (Parts 2-4) take about 1 minute each for 5 epochs on a recent laptop CPU. The language model (Part 5) is much heavier: about 7 minutes on an Apple M4 CPU and considerably longer on older laptops. If it is too slow, run the notebook on Google Colab with a GPU (Runtime > Change runtime type > GPU).

Good luck!